In [1]:
from utils import goto_project_root
from utils.path_settings import MODEL_SAVE_PATH, DATA_PATH, LOG_PATH
from torch.utils.tensorboard import SummaryWriter
import SimulateDatasets.GenTrainingData as g
from utils import create_splits, get_dataloaders, force_remove_dir
from Network_models import Trainer as t
from importlib import reload
reload(t)
reload(g)
import datetime


In [2]:
force_remove_dir('D:/RUNS/runs')
cell_type = "GRU"
configs = {
    "model_specs": {
        "model_name": "CustomRNN", # To diffrentiate from the several types of RNN cells.
        "model_path": "Network_models.RNN_models",
        "model_params": {
            "input_size": 4,
            "hidden_size": 64,
            "num_layers": 1,
            "output_size": 3,
            "cell_type": cell_type,
        }
    },

    "device": "cuda",
    "optimizer_specs": {
        "optimizer_name": "Adam",
        "optimizer_params": {
            "lr": 0.001
        }
    },

    "distance_loss": "geodesic",
    "regularisation_loss": "L2",
    "distance_weight": 1,

    "save_path": MODEL_SAVE_PATH + f"\\{cell_type}_model",
    "log_path": LOG_PATH,
    "check_path": MODEL_SAVE_PATH + f"\\{cell_type}_model_checkpoints",

    "training_config": {
        "task_id": "0.1q",
        "batch_size": 128,
        "seq_len": 1000,
        "prep_phase": 500,
    }
}

configs["training_config"]["data_save_path"] = DATA_PATH + ("\\Data_" + configs["training_config"]["task_id"] + ".pth")
configs["save_path"] = configs["save_path"] + ("_" + configs["training_config"]["task_id"])
configs["check_path"] = configs["check_path"] + ("_" + configs["training_config"]["task_id"])
configs["log_path"] = configs["log_path"] + ("\\Log_" + configs["training_config"]["task_id"] + ".pth")
data1 = g.gen_training_data(configs['training_config'])
dataloaders = get_dataloaders(data1, batch_size=configs["training_config"]["batch_size"], k=5)


Directory does not exist: D:/RUNS/runs


RuntimeError: Parent directory D:\Projects\mental-rotations\data does not exist.

In [ ]:
print('something')

In [4]:
rnn_trainer = t.Trainer(configs)
for i, (train_loader, test_loader) in enumerate(dataloaders):
    print(f"Training on split {i+1}")
    if i:
        rnn_trainer.refresh()
    new_log_path = configs['log_path'] + f"/RNN_model_{configs['training_config']['task_id']}_{i+1}"
    print(new_log_path)
    rnn_trainer.writer = SummaryWriter(new_log_path)
    print(configs["save_path"] + f"_{i+1}")
    print(configs['check_path'] + f"_{i+1}")
    #rnn_trainer.train(train_loader, test_loader, epochs=100, save_path = configs["save_path"] + f"_{i+1}")


NameError: name 'dataloaders' is not defined

In [ ]:
rnn_trainer.model.eval()
for i, (data, target) in enumerate(train_loader):
    data = data.to("cuda")
    target = target.to("cuda")
    output = rnn_trainer.model(data)
    loss = rnn_trainer.loss_fn(rnn_trainer.model.out, rnn_trainer.model.pred, target)
    print(f"total loss: {loss.mean()}") 
    distance_loss = rnn_trainer.distance_loss(rnn_trainer.model.pred, target)
    print(f"distance loss: {distance_loss.mean()}")
    break

In [ ]:
reload(g)
force_remove_dir('D:/RUNS/runs_02q')

configs = {
    "model_specs": {
        "model_name": "CustomRNN", # To diffrentiate from the several types of RNN cells.
        "model_path": "Network_models.RNN_models",
        "model_params": {
            "input_size": 4,
            "hidden_size": 64,
            "num_layers": 1,
            "output_size": 3,
            "cell_type": "GRU",
        }
    },

    "device": "cuda",
    "optimizer_specs": {
        "optimizer_name": "Adam",
        "optimizer_params": {
            "lr": 0.001
        }
    },

    "distance_loss": "geodesic",
    "regularisation_loss": "L2",
    "distance_weight": 1,

    "save_path": MODEL_SAVE_PATH + "RNN_model",
    "log_path": "D:/RUNS/runs_02q",

    "training_config": {
        "task_id": "0.2q",
        "batch_size": 128,
        "seq_len": 1000,
        "prep_phase": 500,
    }
}

configs["training_config"]["save_path"] = DATA_PATH + ("RNN_model_" + configs["training_config"]["task_id"] + ".pth")
data1 = g.gen_training_data(configs['training_config'])
dataloaders = get_dataloaders(data1, batch_size=configs["training_config"]["batch_size"], k=5)
rnn_trainer = t.Trainer(configs)
for i, (train_loader, test_loader) in enumerate(dataloaders):
    print(f"Training on split {i+1}")
    if i:
        rnn_trainer.refresh()
    new_log_path = configs['log_path'] + f"/RNN_model_{configs['training_config']['task_id']}_{i+1}"
    rnn_trainer.writer = SummaryWriter(new_log_path)
    rnn_trainer.train(train_loader, test_loader, epochs=100, save_path = configs["training_config"]["save_path"] + f"_{i+1}")
